In [ ]:
# 🧪 Run LangChain ReAct Agent with Ollama and Evaluate

from langchain_ollama import OllamaLLM
from langchain.agents import create_react_agent, AgentExecutor
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain.evaluation import load_evaluator, EvaluatorType
from datetime import datetime
import os

# 🧠 Initialize Ollama LLM
ollama_llm = OllamaLLM(model="llama3", temperature=0)

# 🔍 Setup Search Tool (DuckDuckGo)
search_tool = DuckDuckGoSearchRun()
tools = [
    Tool(
        name="Search",
        func=search_tool.run,
        description="Useful for answering questions about current events or factual info."
    )
]

# ✍️ Define ReAct Prompt Template
prompt = PromptTemplate.from_template("""You are an AI agent with access to the following tools:

{tools}

Use the format:

Question: the input question you must answer
Thought: what you should do next
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Begin!

Question: {input}
{agent_scratchpad}
""")

# 🤖 Create Agent & Executor
agent = create_react_agent(llm=ollama_llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True, # Prevents crashing on bad format
    max_iterations=6,           # default is 15
    max_execution_time=120       # in seconds
)

# ❓ Question to the Agent
question = "what is the coding best practises?"

# 🚀 Run Agent
response = agent_executor.invoke({"input": question})

# 🔁 Collect Intermediate Steps
intermediate_steps = response.get("intermediate_steps", [])
for step in response.get("intermediate_steps", []):
    print("\n--- Step ---")
    print(step)
    
#from langchain.evaluation import load_evaluator
evaluator = load_evaluator(
    "qa",
    llm=ollama_llm
)
evaluation_result = evaluator.evaluate_strings(
    input=question,
    prediction=response["output"],
    reference="The onboarding process for a new tester in an ongoing project typically includes: introducing them to the team and project, providing tool and process training, assigning initial tasks, and offering mentorship. In Agile setups, this might also involve sprint planning, prioritization, and feedback cycles."
)



# 📝 Save Results to Markdown
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filename = f"evaluation_result_{timestamp}.md"

markdown_lines = [
    "# 🧪 LangChain Agent Evaluation (Ollama)",
    f"**Question:** {question}",
    f"**Final Answer:** {response['output']}",
    "## 🔍 Evaluation Metrics"
]
markdown_lines += [f"- **{key.capitalize()}**: {value}" for key, value in evaluation_result.items()]

with open(filename, "w") as f:
    f.write("\n\n".join(markdown_lines))

# ✅ Output Results
print("\n📝 Evaluation Results:")
for key, value in evaluation_result.items():
    print(f"- {key}: {value}")

print(f"\n✅ Evaluation results saved to `{filename}`.")
print("\n✅ Final Answer:")
print(response["output"])
